# 方案三_v2：图论反应网络动力学评价

> 基于 McDermott et al., *Nature Communications* 2021
> "A graph-based network for predicting chemical reaction pathways in solid-state materials synthesis"

**核心思路**：用热力学自由能 + Softplus 成本函数替代昂贵的 CI-NEB 势垒计算。
构建加权有向反应图，通过 K-最短路径 + 交叉反应 + 线性组合找到最优合成路径。


## 1. 参数配置


In [ ]:
# ============================================================
# ★★★ 用户配置区 — 修改这里即可 ★★★
# ============================================================

# --- 目标化合物 ---
TARGET_FORMULA = "Fe2SiS4"   # pymatgen 约化式（元素按电负性排序，可能与常见书写顺序不同）

# --- 化学体系 ---
# 自动从方案1 scheme1_export.json 同步，无需在此配置

# --- 运行模式 ---
NET_REACTION_KNOWN = True   # True = 已知总反应（多目标KSP）；False = 未知体系（探索模式）

# --- 合成温度 ---
T_SYNTHESIS_C = 627        # 单位：°C（合成温度，四舍五入整数；内部自动转 K）

# --- 热力学过滤（论文推荐 0.03/0.1/0.5 eV/atom）---
E_ABOVE_HULL_CUTOFF = 0.5        # eV/atom，已知总反应时的凸包截断
E_ABOVE_HULL_CUTOFF_UNKNOWN = 0.5  # eV/atom，探索模式截断，包含亚稳中间相
COMPOSITION_WINDOW = False      # 跨体系验证必须关闭，否则会按前驱体比例裁剪掉 Na/S 相

# --- 假设相过滤（论文步骤4）---
FILTER_HYPOTHETICAL = False             # 关闭可避免误删目标/前驱体相关相（若前驱体 KeyError 先检查此项）
HYPOTHETICAL_E_ABOVE_HULL = 0.1         # eV/atom，高于该值视为假设相
EXPERIMENTAL_FORMULAS = []              # 已知实验相白名单，即使能量高也保留，如 ["LiMnO2"]

# --- 前驱体（仅已知模式使用）---
USE_MANUAL_PRECURSORS = True
MANUAL_PRECURSORS = ["Fe5Si3", "Fe3Si", "S"]

# --- 网络构建参数 ---
MAX_REACTANTS = 2        # 最大同时反应相数 n=2 (paper default)
K_SHORTEST = 75          # KSP 路径数（论文值）
KNOWN_BYPRODUCTS = []   # 已知总反应时手动指定副产物；为空则自动推导

# --- 开放元素（巨势）---
USE_OPEN_ELEMENT = False            # True = 交叉反应使用巨势凸包（考虑 OPEN_ELEMENT 化学势）
OPEN_ELEMENT = "O"                  # 开放元素

# --- 探索模式副产物稳定性（可选）---
EXPLORE_CHECK_BYPRODUCT_STABILITY = False  # 是否检查副产物稳定性（论文未作为独立筛选步骤）
EXPLORE_BYPRODUCT_MAX_HULL = 0.1           # 副产物稳定性阈值（仅上述开关为 True 时使用）

# --- 交叉反应与线性组合 ---
ENABLE_CROSSOVER = True   # 启用交叉反应+线性组合
CROSSOVER_MAX_PHASES = 12 # 最多考虑多少中间相做交叉反应
MAX_LINCOMB_CANDIDATES = 60   # 线性组合候选反应数上限
MAX_INTERMEDIATE_RXNS = 15        # 官方 BasicEnumerator 补充的中间相反应数上限（防组合爆炸）
MAX_LINCOMB_COMBO = 6         # 线性组合最大反应步数（论文案例 4~6）
LINCOMB_N_JOBS = -1           # 线性组合并行核数（-1 = 全部核心）
LINCOMB_CHUNK_SIZE = 100000   # 每个并行批次处理的组合数

# --- 成本函数 ---
COST_T_SCALE = 273.0      # 参考温度标度，单位：K（paper 默认 273）

# --- 其他开关 ---
ENABLE_PROXIMITY_FILTER = False  # True = 只保留直接涉及前驱体/目标的反应边
FILTER_INTERDEPENDENT = True    # True = 过滤掉循环依赖的路径 (A->B 且 B->A 互锁)


## 2. 导入库


In [ ]:
import warnings
import os
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import networkx as nx
from itertools import combinations, product as _product
from math import gcd
from collections import defaultdict, Counter
from time import time

import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

from pymatgen.analysis.phase_diagram import PhaseDiagram
from pymatgen.entries.mixing_scheme import MaterialsProjectDFTMixingScheme
from rxn_network.reactions.computed import ComputedReaction
from rxn_network.costs.functions import Softplus
from scipy.linalg import pinv
from scipy.optimize import nnls

print("All imports OK")

# ===== 环境与派生参数（一般无需修改）=====
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # fix OpenMP conflict

T_SYNTHESIS = int(round(T_SYNTHESIS_C + 273.15))   # K（内部使用，取整避免浮点温度键问题）

# 开放元素化学势（当前默认 O）自动从 FactSage 读取，按 T_SYNTHESIS 插值
from rxn_network.data import G_ELEMS as _G_ELEMS
_O2_T = sorted(float(_t) for _t in _G_ELEMS.keys())
_O2_G = [_G_ELEMS[str(int(_t))]["O"] for _t in _O2_T]
MU_O2_DFT_REF = 2.0 * float(np.interp(T_SYNTHESIS, _O2_T, _O2_G))  # eV/O2 (1 atm)


## 3. MP 数据获取与中间相过滤


In [ ]:
import json
from monty.json import MontyDecoder

t0 = time()

# ---- 前驱体元素自动提取（与方案1扩展体系匹配）----
from pymatgen.core import Composition as _Comp
PRECURSOR_ELEMENTS = sorted({str(el) for f in MANUAL_PRECURSORS for el in _Comp(f).elements})
if "TARGET_SYSTEM" not in globals() or not TARGET_SYSTEM:
    TARGET_SYSTEM = []   # 将由方案1 scheme1 元信息同步
EXTENDED_SYSTEM = sorted(set(TARGET_SYSTEM) | set(PRECURSOR_ELEMENTS))
system = EXTENDED_SYSTEM   # 兼容旧代码：计算/导出均使用扩展体系（导出元信息存在时会被覆盖）

# ===== 路线B：方案3 从方案1 0K 条目现场构建 Gibbs（温度由 T_SYNTHESIS 决定）=====
# 方案2 只做多温度稳定性筛选，不再向方案3 提供单一温度导出；
# 方案3 加载 0K 条目后，在 T_SYNTHESIS 下用参考管线（Bartel SISSO + FactSage + NIST）构建 Gibbs。
entries_loaded = False

# --- Priority 2: 方案1 0K 条目（含 MACE-MP-0 预测）---
if not entries_loaded:
    scheme1_export = os.path.join("..", "方案1", "scheme1_export.json")
    if os.path.exists(scheme1_export):
        print("📂 检测到方案1导出文件，从本地加载...")
        try:
            with open(scheme1_export, "r", encoding="utf-8") as f:
                export_data = json.load(f, cls=MontyDecoder)
            exported_system = export_data.get("system", [])
            # 若导出文件带扩展体系元信息，则同步本 Notebook 的体系定义（兼容方案1双体系）
            if "target_system" in export_data:
                TARGET_SYSTEM = list(export_data["target_system"])
                EXTENDED_SYSTEM = list(export_data["extended_system"])
                PRECURSOR_ELEMENTS = list(export_data["precursor_elements"])
                system = EXTENDED_SYSTEM
                print(f"   已同步扩展体系: {system} (目标: {TARGET_SYSTEM})")
            if set(exported_system) != set(system):
                print(f"⚠️ 体系不匹配 (导出: {exported_system}, 当前: {system})，请检查方案1导出文件")
            else:
                entries = export_data["entries"]
                n_mp_s1 = export_data.get("n_entries_mp", "?")
                n_ml_s1 = export_data.get("n_entries_mace", export_data.get("n_entries_m3gnet", "?"))
                print(f"   从方案1加载 {len(entries)} 条条目")
                print(f"   (MP: {n_mp_s1}, MACE-MP-0: {n_ml_s1})")
                for e in entries:
                    if not isinstance(getattr(e, "entry_id", None), str):
                        e.entry_id = str(getattr(e, "entry_id", e.composition.reduced_formula))
                for e in entries:
                    if getattr(e, "data", {}).get("source", "") in ("M3GNet", "MACE-MP-0"):
                        if not hasattr(e, "parameters") or e.parameters is None:
                            e.parameters = {}
                        e.parameters.setdefault("run_type", "M3GNet")
                target_entry = None
                for e in entries:
                    if e.composition.reduced_formula == TARGET_FORMULA:
                        if target_entry is None or e.energy_per_atom < target_entry.energy_per_atom:
                            target_entry = e
                if target_entry is None:
                    raise ValueError(f"Target {TARGET_FORMULA} not found in loaded entries")
                target_struct = target_entry.structure
                target_id = getattr(target_entry, "entry_id", "N/A")
                # 方案1 导出前已对 MP 条目执行过 MaterialsProjectDFTMixingScheme，这里不再重复校正
                entries_loaded = True
        except Exception as e:
            print(f"⚠️ 方案1文件加载失败: {e}，请检查方案1导出文件")
            import traceback
            traceback.print_exc()

# --- 数据必须来自方案1 ---
if not entries_loaded:
    raise FileNotFoundError("未找到方案1的 scheme1_export.json，请先运行 方案1/凸包计算_v2.ipynb")


## 4. 反应网络构建


**节点**：所有不超过 `MAX_REACTANTS` 种的相组合（frozenset of formulas）

**边**：两节点间可化学计量配平的反应。排除恒等反应。


In [ ]:
def enumerate_reactions(entries, max_reactants=2, verbose=True):
    """枚举所有可化学计量配平的反应对"""
    entry_list = list(entries)
    combos = []
    for n in range(1, max_reactants + 1):
        for combo in combinations(entry_list, n):
            combos.append(set(combo))
    if verbose:
        print(f"Phase combinations: {len(combos)}")
    reactions = []
    for i in range(len(combos)):
        for j in range(i + 1, len(combos)):
            if combos[i] & combos[j]:
                continue  # 不能有重叠相
            # 元素集合预检：反应物与产物的元素集合必须相同（元素守恒），否则不可能配平
            r_elems = set().union(*[set(e.composition.elements) for e in combos[i]])
            p_elems = set().union(*[set(e.composition.elements) for e in combos[j]])
            if r_elems != p_elems:
                continue
            try:
                rxn = ComputedReaction.balance(list(combos[i]), list(combos[j]))
                if not rxn.is_identity:
                    reactions.append(rxn)
                    reactions.append(rxn.reverse())
            except Exception:
                pass
    if verbose:
        print(f"Raw reactions: {len(reactions)}")
    return reactions

def deduplicate_by_formula(reactions):
    """按化学式去重"""
    seen = {}
    unique = []
    for rxn in reactions:
        r_key = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        p_key = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        key = (r_key, p_key)
        if key not in seen:
            seen[key] = rxn
            unique.append(rxn)
    return unique

# ---- Enumerate and filter ----
rxns_raw = enumerate_reactions(filtered, MAX_REACTANTS)
rxns_unique = deduplicate_by_formula(rxns_raw)
print(f"After dedup: {len(rxns_unique)} unique reaction edges")

# ---- Proximity filter (optional) ----
if ENABLE_PROXIMITY_FILTER:
    precursor_set = frozenset(PRECURSOR_FORMULAS)
    rxns_filtered = []
    for rxn in rxns_unique:
        r_fs = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        p_fs = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        if (r_fs & precursor_set) or (TARGET_FORMULA in p_fs) or (TARGET_FORMULA in r_fs):
            rxns_filtered.append(rxn)
    rxns_unique = rxns_filtered
    print(f"After proximity filter: {len(rxns_unique)} edges")

N_EDGES = len(rxns_unique)
print(f"\nReaction network: {len(formulas_full)} phases → {N_EDGES} unique reaction edges")


## 5. Softplus 热力学成本函数


**核心公式** (rxn_network costs/functions.py:77):

$$C = \\ln\\left(1 + \\frac{273}{T} \\cdot e^{\\Delta g_{\\text{rxn}}}\\right)$$

其中：
- $T$ = 合成温度 (K)
- $\\Delta g_{\\text{rxn}} =$ `rxn.energy_per_atom`（反应自由能，eV/atom）

**特性**：
- $\\Delta g \\ll 0$（强放热）→ $C \\to 0$
- $\\Delta g > 0$（吸热）→ $C$ 近似 $\\Delta g + \\ln(273/T)$


In [ ]:
# ===== 直接使用 rxn.energy_per_atom（官方 Softplus 的默认参数）=====
# 参考: rxn_network/costs/functions.py: Softplus.evaluate()

def softplus_cost(rxn, T=1073.0):
    """使用 rxn_network 官方 Softplus 计算反应成本。"""
    return float(Softplus(temp=T).evaluate(rxn))


# ===== Compute costs for all edges =====
print("Computing thermodynamic costs for all edges...")
edge_costs = {}  # (r_key, p_key) -> (cost, dg)
for rxn in rxns_unique:
    r_key = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
    p_key = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
    dg = rxn.energy_per_atom  # ← 官方做法：直接用 energy_per_atom
    cost = softplus_cost(rxn, T=T_SYNTHESIS)
    edge_costs[(r_key, p_key)] = (cost, dg)

print(f"Computed costs for {len(edge_costs)} edges")

# ===== Cost statistics =====
costs_list = [c for c, _ in edge_costs.values()]
dgs_list = [d for _, d in edge_costs.values()]
print(f"\nCost statistics:")
print(f"  Δg_rxn range: {min(dgs_list):.4f} ~ {max(dgs_list):.4f} eV/atom")
print(f"  Cost range:   {min(costs_list):.4f} ~ {max(costs_list):.4f}")

# ===== Show top-10 most favorable edges =====
print(f"\nTop-10 most thermodynamically favorable edges:")
sorted_edges = sorted(edge_costs.items(), key=lambda x: x[1][0])
for rank, ((rk, pk), (cost, dg)) in enumerate(sorted_edges[:10]):
    rs = "+".join(sorted(rk))
    ps = "+".join(sorted(pk))
    print(f"  {rank+1}. C={cost:.4f} Δg={dg:+.4f} eV/atom | {rs} → {ps}")


## 6. 图构建：循环边与外部节点


**零成本循环边**：允许生成的中间产物继续参与后续反应，从而捕获多步反应序列。

**外部节点**：
- **SOURCE**：零成本连接到所有前驱体可及的反应物节点
- **SINK**：所有包含目标产物的节点以零成本连接到汇点


In [ ]:
# ---- 提前模式判断（第七点）：探索模式不构建图 ----
EXPLORATION_MODE = False
if not NET_REACTION_KNOWN:
    EXPLORATION_MODE = True
    print("--- 陌生体系探索模式：枚举所有生成目标的反应 ---")
    target_elems = set(entry_map[TARGET_FORMULA].composition.elements)
    candidate_rxns = []
    for rxn in rxns_unique:
        p_rfs = {e.composition.reduced_formula for e in rxn.product_entries}
        if TARGET_FORMULA not in p_rfs:
            continue
        byproducts = p_rfs - {TARGET_FORMULA}
        ok = True
        for bf in byproducts:
            if bf not in entry_map:
                ok = False; break
            b_elems = set(entry_map[bf].composition.elements)
            if b_elems & set(target_elems):   # 副产物含目标元素→排除
                ok = False; break
            if EXPLORE_CHECK_BYPRODUCT_STABILITY:
                hull = entry_map[bf].data.get("e_above_hull", 999)
                if hull > EXPLORE_BYPRODUCT_MAX_HULL:
                    ok = False; break
        if ok:
            candidate_rxns.append(rxn)
    sorted_candidates = sorted(candidate_rxns, key=lambda r: softplus_cost(r, T=T_SYNTHESIS))
    print(f"找到 {len(sorted_candidates)} 个可行反应（过滤后）")
    for i, rxn in enumerate(sorted_candidates[:20]):
        print(f"{i+1}. {rxn}  cost={softplus_cost(rxn, T=T_SYNTHESIS):.4f}")
    ksp_paths_raw = []
    ksp_paths = []

def build_thermo_graph(rxns, edge_costs):
    """构建热力学加权有向反应图 (对标 ReactionNetwork.build)
    - 两类节点: Rx (反应物) 和 Pd (产物)
    - 循环边: Pd → 同组合的 Rx (官方 get_loopback_edges)
    """
    G = nx.DiGraph()
    r_nodes = {}  # frozenset -> idx
    p_nodes = {}
    for rxn in rxns:
        r_key = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        p_key = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        if r_key not in r_nodes:
            idx = len(G.nodes())
            G.add_node(idx, ntype="Rx", formulas=r_key, label="+".join(sorted(r_key)))
            r_nodes[r_key] = idx
        if p_key not in p_nodes:
            idx = len(G.nodes())
            G.add_node(idx, ntype="Pd", formulas=p_key, label="+".join(sorted(p_key)))
            p_nodes[p_key] = idx
        cost, dg = edge_costs.get((r_key, p_key), (10.0, 10.0))
        G.add_edge(r_nodes[r_key], p_nodes[p_key], edge_type="reaction", reaction=rxn, cost=cost, dg=dg)
    # Loopback: product -> same-combo reactant
    lb = 0
    for key in set(r_nodes) & set(p_nodes):
        if not G.has_edge(p_nodes[key], r_nodes[key]):
            G.add_edge(p_nodes[key], r_nodes[key], edge_type="loopback", cost=0.0, dg=0.0)
            lb += 1
    print(f"Graph: {G.number_of_nodes()} nodes ({len(r_nodes)}Rx,{len(p_nodes)}Pd), {G.number_of_edges()} edges ({len(rxns)}rxn+{lb}lb)")
    return G, r_nodes, p_nodes


def add_source_sink(G, r_nodes, p_nodes, source_formulas, target_formula):
    """SOURCE->Rx(subset precursors), Pd(含target)->SINK"""
    SOURCE, SINK = -1, -2
    G.add_node(SOURCE, ntype="Source", label="SOURCE")
    G.add_node(SINK, ntype="Sink", label="SINK")
    sc = set(source_formulas)
    src = snk = 0
    for r_key, r_idx in r_nodes.items():
        if r_key.issubset(sc):
            G.add_edge(SOURCE, r_idx, edge_type="precursor_edge", cost=0.0, dg=0.0)
            src += 1
    for p_key, p_idx in p_nodes.items():
        if target_formula in p_key:
            G.add_edge(p_idx, SINK, edge_type="target_edge", cost=0.0, dg=0.0)
            snk += 1
    print(f"  Source→{src}Rx, {snk}Pd→Sink")
    return SOURCE, SINK


# ===== Build graph =====
if not EXPLORATION_MODE:
    G, r_nodes, p_nodes = build_thermo_graph(rxns_unique, edge_costs)
    SOURCE, SINK = add_source_sink(G, r_nodes, p_nodes, PRECURSOR_FORMULAS, TARGET_FORMULA)
    print(f"Total: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


## 7. K-最短路径搜索（Yen 算法）


使用 Yen's KSP 算法从前驱体节点到目标节点搜索前 K 条最短路径（按累积成本排序）。


In [ ]:
def has_interdependent_rxns(reactions):
    """检查路径中是否有相互依赖的循环 (A->B 和 B->A 互锁)"""
    for i, rxn_i in enumerate(reactions):
        p_i = frozenset(e.composition.reduced_formula for e in rxn_i.product_entries)
        r_i = frozenset(e.composition.reduced_formula for e in rxn_i.reactant_entries)
        for j in range(i + 1, len(reactions)):
            p_j = frozenset(e.composition.reduced_formula for e in reactions[j].product_entries)
            r_j = frozenset(e.composition.reduced_formula for e in reactions[j].reactant_entries)
            if (p_i & r_j) and (p_j & r_i):
                return True
    return False


def yen_ksp(G, source, target, K=10):
    """Yen's K-Shortest Paths（保留给分解报告使用；合成搜索改用官方网络）"""
    try:
        fp = nx.shortest_path(G, source, target, weight="cost")
        fc = sum(G[fp[i]][fp[i + 1]]["cost"] for i in range(len(fp) - 1))
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return []
    A = [(fc, fp)]
    B = []
    for k in range(1, K):
        prev = A[-1][1]
        for i in range(len(prev) - 1):
            spur = prev[i]
            root = prev[:i + 1]
            Gc = G.copy()
            for _, pv in A:
                if len(pv) > i + 1 and pv[:i + 1] == root:
                    u, v = pv[i], pv[i + 1]
                    if Gc.has_edge(u, v):
                        Gc.remove_edge(u, v)
            for node in root[:-1]:
                if node != spur and node >= 0:
                    Gc.remove_node(node)
            try:
                sp = nx.shortest_path(Gc, spur, target, weight="cost")
                tp = root[:-1] + sp
                tc = sum(G[tp[j]][tp[j + 1]]["cost"] for j in range(len(tp) - 1)
                         if G.has_edge(tp[j], tp[j + 1]))
                if tc > 0:
                    B.append((tc, tp))
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                pass
        if not B:
            break
        B.sort(key=lambda x: x[0])
        found = False
        new_B = []
        for cv, pv in B:
            if pv not in [p for _, p in A]:
                A.append((cv, pv))
                found = True
            else:
                new_B.append((cv, pv))
        B = new_B
        if not found:
            break
    return A


def extract_pathway(G, path):
    """兼容层：支持官方 BasicPathway 路径对象（也保留旧 node_path 逻辑）"""
    if isinstance(path, tuple):
        path = path[1]
    if hasattr(path, "reactions"):
        rxns = list(path.reactions)
        costs = list(path.costs) if len(path.costs) == len(rxns) else [0.0] * len(rxns)
        dgs = [rxn.energy_per_atom for rxn in rxns]
        return rxns, costs, dgs
    # 旧 node_path 兼容（用于分解图 G_dec）
    rxns, costs, dgs = [], [], []
    for i in range(len(path) - 1):
        edge = G[path[i]][path[i + 1]]
        rxn = edge.get("reaction")
        if rxn is not None and not isinstance(rxn, str):
            rxns.append(rxn)
            costs.append(edge.get("cost", 0))
            dgs.append(edge.get("dg", 0))
    return rxns, costs, dgs


# ===== 官方 rxn_network 搜索（论文做法）=====
if not EXPLORATION_MODE:
    from rxn_network.network.network import ReactionNetwork
    from rxn_network.reactions.reaction_set import ReactionSet
    from rxn_network.costs.functions import Softplus

    precursor_set = set(PRECURSOR_FORMULAS)
    byproduct_candidates = set()
    for rxn in rxns_unique:
        r_fs = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        p_fs = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        if r_fs.issubset(precursor_set) and not (p_fs & precursor_set):
            byproduct_candidates.update(p_fs)
    # 论文做法：KSP 只搜索已知目标（目标 + 手动指定副产物）；未指定副产物时只搜目标
    # （byproduct_candidates 仍保留，供 total_rxn 配平时自动推导使用）
    KSP_TARGETS = [TARGET_FORMULA] + list(KNOWN_BYPRODUCTS)
    print(f"KSP targets: {KSP_TARGETS}")

    cf = Softplus(temp=T_SYNTHESIS)
    network = ReactionNetwork(ReactionSet.from_rxns(rxns_unique), cf)
    print("构建官方 ReactionNetwork...")
    network.build()
    network.set_precursors(list(PRECURSOR_FORMULAS))
    pathway_set = network.find_pathways(KSP_TARGETS, k=K_SHORTEST)
    ksp_paths_raw = [(p.total_cost, p) for p in list(pathway_set)]
    print(f"[KSP] 官方搜索 raw: {len(ksp_paths_raw)} paths")

    ksp_paths = [(tc, p) for tc, p in ksp_paths_raw if len(p.reactions) > 0]
    if FILTER_INTERDEPENDENT:
        ksp_paths = [(tc, p) for tc, p in ksp_paths
                     if not has_interdependent_rxns(p.reactions)]
        print(f"KSP after interdependency filter: {len(ksp_paths)} paths")
    print(f"KSP: {len(ksp_paths)} synthesis paths found (filtered from {len(ksp_paths_raw)} raw)")

    print("| 路径 | 总成本 | 步骤 | 反应方程式 | ΔG (eV/atom) | 步骤成本 |")
    print("| :---: | :---: | :---: | :--- | :---: | :---: |")
    for rank, (tc, p) in enumerate(ksp_paths[:5]):
        rxns_p, costs_p, dgs_p = extract_pathway(None, p)
        for j, rxn in enumerate(rxns_p):
            print(f"| {rank + 1} | {tc:.3f} | {j + 1} | {rxn} | {dgs_p[j]:+.3f} | {costs_p[j]:.3f} |")


## 8. 交叉反应与线性组合


**交叉反应**（Crossover Reactions）：从 KSP 路径中提取中间相，枚举它们之间所有能生成目标产物的反应。

**线性组合**：将所有交叉反应按化学计量线性组合，求解满足总反应（前驱体 → 目标）的最优反应倍数。
总成本 = $\frac{\sum m_i \cdot C_i}{\sum m_i}$（加权平均）。


In [ ]:
def extract_intermediate_phases(ksp_paths, G=None, top_k=K_SHORTEST):
    """从KSP前top_k路径中提取中间相（单相）——兼容官方 BasicPathway"""
    intermediates = set()
    for _, path in ksp_paths[:top_k]:
        rxns, _, _ = extract_pathway(G, path)
        for rxn in rxns:
            for e in rxn.entries:
                f = e.composition.reduced_formula
                if f != TARGET_FORMULA and f not in PRECURSOR_FORMULAS:
                    intermediates.add(f)
    return intermediates


def compute_crossover_reactions(filtered_entries, entry_map, precursors, target, intermediates, pd=None, chempots=None):
    """计算交叉反应：
    - 简单组合法（保留）：reactants → target + other phases
    - 相图法（新增）：用 PhaseDiagram.get_decomposition 预测最稳定产物（可引入 CO2 等副产物）
    """
    candidate_formulas = set(precursors) | intermediates
    candidate_formulas.discard(target)
    candidate_entries = [entry_map[f] for f in candidate_formulas if f in entry_map]
    target_entry = entry_map.get(target)
    if target_entry is None or len(candidate_entries) < 1:
        return []

    # Build all 1- and 2-phase combinations as reactants
    combos = []
    # Single-phase reactants
    for e in candidate_entries:
        combos.append(set([e]))
    # Two-phase combinations
    for combo in combinations(candidate_entries, 2):
        combos.append(set(combo))

    crossover_rxns = []
    target_elems = set(target_entry.composition.elements)
    for reactant_set in combos:
        reactant_list = list(reactant_set)
        r_elems = set().union(*[set(e.composition.elements) for e in reactant_list])
        # Try: reactants → target only（元素集合必须相同）
        if r_elems == target_elems:
            try:
                rxn = ComputedReaction.balance(reactant_list, [target_entry])
                if not rxn.is_identity:
                    crossover_rxns.append(rxn)
            except Exception:
                pass

        # 相图法：用凸包分解预测最稳定产物集合（自动引入 CO2、LiCl 等副产物）
        if pd is not None:
            try:
                total_comp = reactant_list[0].composition
                for e in reactant_list[1:]:
                    total_comp = total_comp + e.composition
                decomp = pd.get_decomposition(total_comp)
                reactant_rfs = {e.composition.reduced_formula for e in reactant_list}
                products = [e for e in decomp if e.composition.reduced_formula not in reactant_rfs]
                if products:
                    if chempots:
                        rxn = OpenComputedReaction.balance(reactant_list, products, chempots)
                    else:
                        rxn = ComputedReaction.balance(reactant_list, products)
                    if not rxn.is_identity:
                        crossover_rxns.append(rxn)
            except Exception:
                pass

        # Try: reactants → target + other phases（简单组合法，保留）
        # Use all entries as possible co-products
        for other_set in combos:
            if reactant_set & other_set:
                continue
            other_entries = [e for e in other_set
                             if e.composition.reduced_formula != target]
            if not other_entries:
                continue
            # 元素集合预检：反应物元素 == 产物（target + others）元素
            p_elems = set(target_elems)
            for e in other_entries:
                p_elems.update(e.composition.elements)
            if r_elems != p_elems:
                continue
            try:
                products = [target_entry] + other_entries
                rxn = ComputedReaction.balance(reactant_list, products)
                if not rxn.is_identity:
                    crossover_rxns.append(rxn)
            except Exception:
                pass

    # Deduplicate
    seen = set()
    unique_rxns = []
    for rxn in crossover_rxns:
        rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
        pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
        key = (rk, pk)
        if key not in seen:
            seen.add(key)
            unique_rxns.append(rxn)
    return unique_rxns


if ENABLE_CROSSOVER and not EXPLORATION_MODE:
    # 从 KSP 路径提取中间相（若有）；否则用全部非前驱体/非目标相
    if len(ksp_paths) > 0:
        print("--- Extracting intermediates from top KSP paths ---")
        intermediates = extract_intermediate_phases(ksp_paths, G, top_k=K_SHORTEST)
    else:
        print("--- KSP empty, using all non-precursor/target phases as intermediates ---")
        intermediates = set(formulas_full) - set(PRECURSOR_FORMULAS) - {TARGET_FORMULA}
    print(f"Intermediate phases: {sorted(intermediates) if intermediates else '(none)'}")

    print(f"\n--- Computing crossover reactions ---")
    # 构建交叉反应用的凸包（温度修正后的网络条目）
    # 开放元素模式：使用巨势凸包（GrandPotentialPhaseDiagram）考虑 OPEN_ELEMENT 化学势
    if USE_OPEN_ELEMENT:
        from pymatgen.analysis.phase_diagram import GrandPotentialPhaseDiagram
        from pymatgen.core import Element as _PmgElement
        from rxn_network.reactions.open import OpenComputedReaction
        chempots = {_PmgElement(OPEN_ELEMENT): MU_O2_DFT_REF / 2.0}  # 单原子化学势
        pd_cross = GrandPotentialPhaseDiagram(list(entry_map.values()), chempots=chempots)
    else:
        chempots = None
        pd_cross = PhaseDiagram(list(entry_map.values()))
    crossover_rxns = compute_crossover_reactions(
        filtered, entry_map, PRECURSOR_FORMULAS, TARGET_FORMULA, intermediates, pd=pd_cross, chempots=chempots)
    print(f"Crossover reactions: {len(crossover_rxns)}")

    if crossover_rxns:
        # Compute costs for crossover reactions
        crossover_edge_costs = {}
        for rxn in crossover_rxns:
            rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
            pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
            dg = rxn.energy_per_atom
            cost = softplus_cost(rxn, T=T_SYNTHESIS)
            crossover_edge_costs[(rk, pk)] = (cost, dg, rxn)

        # Show top crossover reactions by cost
        print(f"\nTop crossover reactions (by cost):")
        sorted_cross = sorted(crossover_edge_costs.items(), key=lambda x: x[1][0])
        for rank, ((rk, pk), (cost, dg, rxn)) in enumerate(sorted_cross[:15]):
            rs = "+".join(sorted(rk))
            ps = "+".join(sorted(pk))
            print(f"  {rank+1}. C={cost:.4f} Δg={dg:+.4f} | {rs} → {ps}")

        print(f"\n--- Linear combination (pseudo-inverse) ---")
        # Build total reaction: precursors → target
        precursor_entries = [entry_map[f] for f in PRECURSOR_FORMULAS if f in entry_map]
        target_entry = entry_map.get(TARGET_FORMULA)

        if target_entry and len(precursor_entries) == len(PRECURSOR_FORMULAS):
            # ---- 构造总反应（含副产物）：precursors → target + byproducts ----
            from itertools import combinations as _combos_net
            total_rxn = None
            # 优先使用手动指定的副产物构造总反应（与 KNOWN_BYPRODUCTS 一致）
            if KNOWN_BYPRODUCTS:
                byproduct_entries = [entry_map[bf] for bf in KNOWN_BYPRODUCTS if bf in entry_map]
                if byproduct_entries:
                    try:
                        _tr = ComputedReaction.balance(
                            precursor_entries, [target_entry] + byproduct_entries)
                        if not _tr.is_identity:
                            total_rxn = _tr
                    except Exception:
                        pass
            if total_rxn is None:
                try:
                    _tr = ComputedReaction.balance(precursor_entries, [target_entry])
                    if not _tr.is_identity:
                        total_rxn = _tr
                except Exception:
                    pass
            if total_rxn is None:
                # 前驱体含目标没有的元素（如 Li/C/Cl）时，需要副产物参与才能配平
                byproduct_pool = []
                for rxn in rxns_unique:
                    r_fs = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
                    p_fs = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
                    if r_fs.issubset(set(PRECURSOR_FORMULAS)) and not (p_fs & set(PRECURSOR_FORMULAS)):
                        for f in p_fs:
                            if f != TARGET_FORMULA and f in entry_map:
                                byproduct_pool.append(entry_map[f])
                # 去重（按 reduced_formula）
                _seen_bp = {}
                for e in byproduct_pool:
                    _seen_bp[e.composition.reduced_formula] = e
                byproduct_pool = list(_seen_bp.values())
                for n_prod in range(1, 4):
                    if total_rxn is not None:
                        break
                    for prod_combo in _combos_net(byproduct_pool, n_prod):
                        try:
                            _tr = ComputedReaction.balance(
                                precursor_entries, [target_entry] + list(prod_combo))
                            if not _tr.is_identity:
                                total_rxn = _tr
                                break
                        except Exception:
                            continue
            if total_rxn is not None and not total_rxn.is_identity:
                print(f"Total reaction: {total_rxn}")
                # Build total stoichiometry dict
                total_stoich = {}
                all_ents2 = tuple(total_rxn.reactant_entries) + tuple(total_rxn.product_entries)
                for coeff, entry in zip(total_rxn.coefficients, all_ents2):
                    rf = entry.composition.reduced_formula
                    total_stoich[rf] = total_stoich.get(rf, 0) + coeff

                # Collect all phases involved in crossover reactions
                all_phases = set(total_stoich.keys())
                for (rk, pk) in crossover_edge_costs:
                    all_phases.update(rk); all_phases.update(pk)
                all_phases = sorted(all_phases)
                phase_idx = {p: i for i, p in enumerate(all_phases)}

                # Build target vector c and matrix A
                n_phases = len(all_phases)
                c = np.zeros(n_phases)
                for formula, coeff in total_stoich.items():
                    c[phase_idx[formula]] = coeff

                # ===== 论文式线性组合：Numba 批量配平 + joblib 并行（保留论文加速，绕开 Windows Ray 崩溃）=====
                import math as _math
                from itertools import combinations as _combos
                from rxn_network.pathways.solver import _balance_path_arrays_cpu   # 纯 Numba 内核（官方代码）
                from rxn_network.pathways.balanced import BalancedPathway
                from joblib import Parallel, delayed

                # 1) 候选反应：只取“生成目标”的 KSP 路径反应（论文：候选集来自到目标的 k 最短路径）
                ksp_paths_target = [
                    (_tc, _path) for _tc, _path in ksp_paths
                    if TARGET_FORMULA in {e.composition.reduced_formula
                                          for e in _path.reactions[-1].product_entries}
                ]
                print(f"  KSP paths to target: {len(ksp_paths_target)} (of {len(ksp_paths)} total)")
                seen_rxn = set()
                candidate_rxns = []
                for _tc, _path in ksp_paths_target:
                    for _rxn in extract_pathway(G, _path)[0]:
                        _rk = frozenset(e.composition.reduced_formula for e in _rxn.reactant_entries)
                        _pk = frozenset(e.composition.reduced_formula for e in _rxn.product_entries)
                        _key = (_rk, _pk)
                        if _key not in seen_rxn:
                            seen_rxn.add(_key)
                            candidate_rxns.append(_rxn)
                # 1.5) 官方 _find_intermediate_rxns 的 Ray-free 实现：枚举中间相之间的反应并补入候选
                #      BasicEnumerator 依赖 Ray（本机 Windows 崩溃风险高），这里用 ComputedReaction.balance 等价实现

                _interm_entries = set()
                for _rxn in candidate_rxns:
                    _interm_entries.update(_rxn.entries)
                _interm_entries.add(entry_map[TARGET_FORMULA])
                _ref_elems = {e for e in gibbs_set.entries if e.is_element}
                _interm_entries.update(_ref_elems)
                _interm_list = sorted(_interm_entries, key=lambda e: e.composition.reduced_formula)

                # 枚举 1~2 反应物 × 1~2 产物（全部在中间相集内），与官方 BasicEnumerator 同思路
                _interm_rxns = []
                _seen_tmp = set()
                _combos1 = [(e,) for e in _interm_list]
                _combos2 = list(combinations(_interm_list, 2))
                _react_combos = _combos1 + _combos2
                _prod_combos = _react_combos
                for _rs in _react_combos:
                    _r_elems = set().union(*[set(e.composition.elements) for e in _rs])
                    for _ps in _prod_combos:
                        if set(_rs) & set(_ps):
                            continue
                        _p_elems = set().union(*[set(e.composition.elements) for e in _ps])
                        if _r_elems != _p_elems:
                            continue
                        try:
                            _rxn = ComputedReaction.balance(list(_rs), list(_ps))
                            if _rxn.is_identity:
                                continue
                            _rk = frozenset(e.composition.reduced_formula for e in _rxn.reactant_entries)
                            _pk = frozenset(e.composition.reduced_formula for e in _rxn.product_entries)
                            _key = (_rk, _pk)
                            if _key in _seen_tmp:
                                continue
                            _seen_tmp.add(_key)
                            if _rxn.energy_per_atom < 0.0:   # 官方默认只保留放热反应
                                _interm_rxns.append(_rxn)
                        except Exception:
                            pass
                print(f"  Intermediate reactions (Ray-free): {len(_interm_rxns)} exergonic")
                _interm_rxns = sorted(_interm_rxns, key=lambda r: softplus_cost(r, T=T_SYNTHESIS))
                _added = 0
                for _rxn in _interm_rxns:
                    if _added >= MAX_INTERMEDIATE_RXNS:
                        break
                    _rk = frozenset(e.composition.reduced_formula for e in _rxn.reactant_entries)
                    _pk = frozenset(e.composition.reduced_formula for e in _rxn.product_entries)
                    _key = (_rk, _pk)
                    if _key not in seen_rxn:
                        seen_rxn.add(_key)
                        candidate_rxns.append(_rxn)
                        _added += 1
                print(f"  Added {_added} intermediate reactions (Ray-free, cap={MAX_INTERMEDIATE_RXNS})")


                sorted_cross = sorted(crossover_edge_costs.items(), key=lambda x: x[1][0])
                for (rk, pk), (cost, dg, rxn) in sorted_cross:
                    if len(candidate_rxns) >= MAX_LINCOMB_CANDIDATES:
                        break
                    if (rk, pk) not in seen_rxn:
                        seen_rxn.add((rk, pk))
                        candidate_rxns.append(rxn)
                print(f"  Linear-combination candidates: {len(candidate_rxns)} reactions "
                      f"(KSP-path reactions + top crossover, cap={MAX_LINCOMB_CANDIDATES})")
                net_phases = set(total_stoich.keys())
                candidate_rxns = [rxn for rxn in candidate_rxns
                                  if any(e.composition.reduced_formula in net_phases
                                         for e in list(rxn.reactant_entries) + list(rxn.product_entries))]
                print(f"  After zero-contribution filter: {len(candidate_rxns)} reactions")
                # 论文做法：候选集 = KSP 路径反应（全部保留）+ 交叉反应（填到 cap）
                # 不再按单个反应成本截断 KSP 反应，否则会丢掉总反应本身导致 0 条配平
                # 确保总反应本身一定在候选里（n=1 即可配平）
                _tr_rk = frozenset(e.composition.reduced_formula for e in total_rxn.reactant_entries)
                _tr_pk = frozenset(e.composition.reduced_formula for e in total_rxn.product_entries)
                _tr_key = (_tr_rk, _tr_pk)
                if _tr_key not in seen_rxn:
                    seen_rxn.add(_tr_key)
                    candidate_rxns.append(total_rxn)
                print(f"  Final candidate reactions: {len(candidate_rxns)} (incl. net reaction)")

                # 2) 用“相(化学式)基”构建反应向量（不依赖 entry.data['idx']，比官方更稳）
                all_phases = set(total_stoich.keys())
                for rxn in candidate_rxns:
                    for e in list(rxn.reactant_entries) + list(rxn.product_entries):
                        all_phases.add(e.composition.reduced_formula)
                all_phases = sorted(all_phases)
                phase_idx = {p: i for i, p in enumerate(all_phases)}
                n_phases = len(all_phases)

                net_vec = np.zeros(n_phases)
                for formula, coeff in total_stoich.items():
                    net_vec[phase_idx[formula]] = coeff

                def _rxn_vec(rxn):
                    v = np.zeros(n_phases)
                    # 用 rxn.entries 而不是 reactant_entries+product_entries：
                    # 官方 get_entry_idx_vector 也是用 self.entries 与 coefficients 对齐
                    for coeff, e in zip(rxn.coefficients, rxn.entries):
                        v[phase_idx[e.composition.reduced_formula]] += coeff
                    return v

                reaction_vecs = np.array([_rxn_vec(rxn) for rxn in candidate_rxns])
                cross_costs = []
                for rxn in candidate_rxns:
                    rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
                    pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
                    _item = crossover_edge_costs.get((rk, pk))
                    if _item is None:
                        cross_costs.append(softplus_cost(rxn, T=T_SYNTHESIS))
                    else:
                        cross_costs.append(_item[0])
                costs_arr = np.array(cross_costs)


                # 3) 复刻官方 _create_comp_matrices：把一组组合堆成 (batch, n, n_phases) 矩阵并预筛
                net_filter = np.argwhere(np.abs(net_vec) > 1e-6).flatten()

                def _build_comp_matrices(combos):
                    if not combos:
                        return np.empty((0, 0, n_phases), dtype=np.float64)
                    cm = np.stack([np.stack([reaction_vecs[r] for r in combo]) for combo in combos])
                    return cm[cm[:, :, net_filter].any(axis=1).all(axis=1)]

                def _process_chunk(combos):
                    """一个分片：构建矩阵 + 官方 Numba 批量配平（对应 _balance_path_arrays_cpu）"""
                    cm = _build_comp_matrices(combos)
                    if cm.shape[0] == 0:
                        return []
                    c_mats, m_mats = _balance_path_arrays_cpu(cm, net_vec)
                    return list(zip(c_mats, m_mats))

                def _chunked(iterable, size):
                    chunk = []
                    for item in iterable:
                        chunk.append(item)
                        if len(chunk) >= size:
                            yield chunk
                            chunk = []
                    if chunk:
                        yield chunk

                # 4) 分块枚举组合 + joblib 并行（Windows 兼容，替代 Ray）
                print(f"  Numba+joblib linear combination: max_combo={MAX_LINCOMB_COMBO}, "
                      f"candidates={len(candidate_rxns)}, chunk={LINCOMB_CHUNK_SIZE}, n_jobs={LINCOMB_N_JOBS}")
                balanced_raw = []
                n_candidates = len(candidate_rxns)
                for n in range(1, MAX_LINCOMB_COMBO + 1):
                    n_combos = _math.comb(n_candidates, n)
                    print(f"    Searching n={n}: {n_combos} combos")
                    # 惰性分块：不能 list() 化全部组合，否则 5000 万组合会 MemoryError
                    # joblib 直接消费生成器，逐块提交（对应官方 grouper 惰性分块）
                    results = Parallel(n_jobs=LINCOMB_N_JOBS, backend="loky")(
                        delayed(_process_chunk)(chunk)
                        for chunk in _chunked(_combos(range(n_candidates), n), LINCOMB_CHUNK_SIZE))
                    for res in results:
                        balanced_raw.extend(res)

                # 5) 重建 BalancedPathway（对应官方重建逻辑）
                balanced_paths = []
                for c_mat, m_mat in balanced_raw:
                    path_rxns = []
                    path_costs = []
                    for rxn_mat in c_mat:
                        idxs = np.flatnonzero(np.abs(rxn_mat) > 1e-8)
                        ents = [entry_map[all_phases[i]] for i in idxs]
                        coeffs = [rxn_mat[i] for i in idxs]
                        rxn = ComputedReaction(entries=ents, coefficients=coeffs)
                        path_rxns.append(rxn)
                        rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
                        pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
                        _item = crossover_edge_costs.get((rk, pk))
                        if _item is not None:
                            path_costs.append(_item[0])
                        else:
                            path_costs.append(softplus_cost(rxn, T=T_SYNTHESIS))
                    bp = BalancedPathway(path_rxns, list(m_mat), path_costs, balanced=True)
                    balanced_paths.append(bp)

                # 去重 + 排序 + 互依赖过滤（对应官方 solve 的收尾）
                balanced_paths = sorted(set(balanced_paths), key=lambda p: p.average_cost)
                if FILTER_INTERDEPENDENT:
                    precursor_comps = [e.composition for e in total_rxn.reactant_entries]
                    balanced_paths = [p for p in balanced_paths
                                      if not p.contains_interdependent_rxns(precursor_comps)]
                print(f"  Found {len(balanced_paths)} balanced combinations\n")
                if balanced_paths:
                    for pi, bp in enumerate(balanced_paths[:5]):
                        print(f"  --- Path {pi+1} (cost={bp.average_cost:.4f}) ---")
                        for j, (rxn, mj) in enumerate(zip(bp.reactions, bp.coefficients)):
                            print(f"    {j+1}. x{mj:.3f} | {rxn}")
                        print()
                else:
                    print("  No balanced combination found")
        else:
            print("Cannot balance total reaction: precursor/target entry not found")
else:
    print("Crossover + linear combination SKIPPED")


## 9. 分解反应路径


逆方向：从目标产物出发，搜索分解为前驱体相的路径，评估材料的热力学分解倾向。


In [ ]:
if not EXPLORATION_MODE:
    # ===== Build reverse graph (decomposition) =====
    G_dec = G.copy()
    # 移除所有回边 —— 分解路径不应绕行
    edges_to_remove = [(u,v) for u,v,d in G_dec.edges(data=True) if d.get("edge_type") == "loopback"]
    G_dec.remove_edges_from(edges_to_remove)
    for old in [-1, -2]:
        if old in G_dec.nodes():
            G_dec.remove_node(old)
    SOURCE_DEC, SINK_DEC = -1, -2
    G_dec.add_node(SOURCE_DEC, ntype="Source", label="SOURCE_DEC")
    G_dec.add_node(SINK_DEC, ntype="Sink", label="SINK_DEC")
    src_dec = snk_dec = 0
    pset = set(PRECURSOR_FORMULAS)
    for r_key, r_idx in r_nodes.items():
        if TARGET_FORMULA in r_key:
            G_dec.add_edge(SOURCE_DEC, r_idx, edge_type="precursor_edge", cost=0.0, dg=0.0)
            src_dec += 1
    for p_key, p_idx in p_nodes.items():
        if p_key.issubset(pset):
            G_dec.add_edge(p_idx, SINK_DEC, edge_type="target_edge", cost=0.0, dg=0.0)
            snk_dec += 1
    print(f"Decomposition: Target→{src_dec}Rx, {snk_dec}Pd→Precursors")

    # ===== KSP for decomposition =====
    ksp_dec_raw = yen_ksp(G_dec, SOURCE_DEC, SINK_DEC, K=K_SHORTEST)
    if FILTER_INTERDEPENDENT:
        ksp_dec_raw = [(tc, p) for tc, p in ksp_dec_raw
                       if not has_interdependent_rxns(extract_pathway(G_dec, p)[0])]
        print(f"Decomposition KSP after interdependency: {len(ksp_dec_raw)} raw paths")
    # 过滤: 第一条反应必须真正消耗目标 (系数 < -1e-10) (系数 < 0)
    ksp_dec = []
    for tc, p in ksp_dec_raw:
        rxns = extract_pathway(G_dec, p)[0]
        if not rxns:
            continue
        rxn = rxns[0]
        # 用 rxn.entries 与 coefficients 对齐（reactant_entries+product_entries 顺序不可靠）
        consumed = any(
            coeff < -1e-10 and e.composition.reduced_formula == TARGET_FORMULA
            for coeff, e in zip(rxn.coefficients, rxn.entries)
        )
        if consumed:
            ksp_dec.append((tc, p))
    print(f"Decomposition KSP: {len(ksp_dec)} paths (filtered from {len(ksp_dec_raw)} raw)")

    # ===== Display decomposition pathways as markdown table =====
    print("\n| 路径 | 总成本 | 步骤 | 反应方程式 | ΔG (eV/atom) | 步骤成本 |")
    print("| :---: | :---: | :---: | :--- | :---: | :---: |")
    for rank, (tc, path) in enumerate(ksp_dec[:5]):
        rxns_p, costs_p, dgs_p = extract_pathway(G_dec, path)
        for j, rxn in enumerate(rxns_p):
            rk = frozenset(e.composition.reduced_formula for e in rxn.reactant_entries)
            pk = frozenset(e.composition.reduced_formula for e in rxn.product_entries)
            print(f"| {rank+1} | {tc:.3f} | {j+1} | {rxn} | {dgs_p[j]:+.3f} | {costs_p[j]:.3f} |")
        print()  # blank line after table


## 10. 可视化：合成与分解路径


累积成本（热力学可行性评分）vs 反应步数。
成本越低 = 热力学越有利。注意：这评价的是热力学驱动力而非显式活化能。


In [ ]:
if not EXPLORATION_MODE:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- Synthesis paths ---
    ax = axes[0]
    labels_syn = []  # init outside loop for empty ksp_paths
    for rank, (cost, path) in enumerate(ksp_paths[:3]):
        labels_syn = []
        rxns_p, costs_p, dgs_p = extract_pathway(G, path)
        for rxn in rxns_p:
            labels_syn.append(str(rxn))
        rxns_p, costs_p, dgs_p = extract_pathway(G, path)
        cum = np.cumsum([0] + costs_p)
        ax.plot(range(len(cum)), cum, "-o", lw=2, ms=8,
                label=f"Path {rank+1} (∑C={cost:.3f})")
    ax.axhline(0, color="gray", ls="--")
    if labels_syn:
        ax.set_xticks(range(len(labels_syn)))
        ax.set_xticklabels(labels_syn, fontsize=7, rotation=30, ha="right")
    ax.set_ylabel("Cumulative Softplus Cost")
    ax.set_title("Synthesis: " + "+".join(PRECURSOR_FORMULAS) + " → " + TARGET_FORMULA)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # --- Decomposition paths ---
    ax = axes[1]
    labels_dec = []  # init outside loop
    for rank, (cost, path) in enumerate(ksp_dec[:3]):
        labels_dec = []
        for n in path:
            if n >= 0:
                labels_dec.append(G_dec.nodes[n].get("label", str(n)))
        rxns_p, costs_p, dgs_p = extract_pathway(G_dec, path)
        cum = np.cumsum([0] + costs_p)
        ax.plot(range(len(cum)), cum, "-s", lw=2, ms=8,
                label=f"Path {rank+1} (∑C={cost:.3f})")
    ax.axhline(0, color="gray", ls="--")
    if labels_dec:
        ax.set_xticks(range(len(labels_dec)))
        ax.set_xticklabels(labels_dec, fontsize=7, rotation=30, ha="right")
    ax.set_ylabel("Cumulative Softplus Cost")
    ax.set_title("Decomposition: " + TARGET_FORMULA + " → " + "+".join(PRECURSOR_FORMULAS))
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout(); plt.show()


## 11. 汇总输出与报告保存


In [ ]:
print("=" * 70)
print(f"  Thermodynamic Reaction Network + KSP Evaluation")
print(f"  Based on McDermott et al., Nat. Commun. 2021")
print(f"  Target: {TARGET_FORMULA} ({target_id})")
print(f"  Precursors: {PRECURSOR_FORMULAS}")
print(f"  T_synthesis: {T_SYNTHESIS} K")
print(f"  Cost function: C = ln(1 + (273/{T_SYNTHESIS}) * exp(Δg))")
print("=" * 70)

print(f"\n  Chemical system: {els}")
print(f"  Phase filtering: e_above_hull < {effective_hull_cutoff} eV/atom → {len(formulas_full)} phases")
print(f"  Reaction edges: {N_EDGES} unique")
if not EXPLORATION_MODE:
    print(f"  Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ---- Synthesis ----
if EXPLORATION_MODE:
    print("\n## Exploration mode: candidate reactions producing target")
    for i, rxn in enumerate(sorted_candidates[:20]):
        print(f"{i+1}. {rxn}  cost={softplus_cost(rxn, T=T_SYNTHESIS):.4f}")
    print()
elif len(ksp_paths) > 0:
    if "balanced_paths" in globals() and balanced_paths:
        best_syn = balanced_paths[0]
        print(f"\n## Best synthesis pathway (weighted average cost={best_syn.average_cost:.4f})")
        print("| 步骤 | 反应方程式 | ΔG (eV/atom) | 成本 | 倍数 |")
        print("| :---: | :--- | :---: | :---: | :---: |")
        for j, rxn in enumerate(best_syn.reactions):
            print(f"| {j+1} | {rxn} | {rxn.energy_per_atom:+.3f} | {best_syn.costs[j]:.3f} | {best_syn.coefficients[j]:.3f} |")
        print()
    else:
        best_syn = ksp_paths[0]
        rxns_best, costs_best, dgs_best = extract_pathway(G, best_syn[1])
        print(f"\n## Best synthesis pathway (total_cost={best_syn[0]:.4f})")
        print("| 步骤 | 反应方程式 | ΔG (eV/atom) | 步骤成本 |")
        print("| :---: | :--- | :---: | :---: |")
        for j, rxn in enumerate(rxns_best):
            print(f"| {j+1} | {rxn} | {dgs_best[j]:+.3f} | {costs_best[j]:.3f} |")
        print()

# ---- Decomposition ----
if not EXPLORATION_MODE and len(ksp_dec) > 0:
    best_dec = ksp_dec[0]
    rxns_dec, costs_dec, dgs_dec = extract_pathway(G_dec, best_dec[1])
    print(f"\n## Best decomposition pathway (total_cost={best_dec[0]:.4f})")
    print("| 步骤 | 反应方程式 | ΔG (eV/atom) | 步骤成本 |")
    print("| :---: | :--- | :---: | :---: |")
    for j, rxn in enumerate(rxns_dec):
        print(f"| {j+1} | {rxn} | {dgs_dec[j]:+.3f} | {costs_dec[j]:.3f} |")
    print()

# ---- Save report ----
with open("kinetics_network_v2_report.md", "w", encoding="utf-8") as f:
    pfx = " + ".join(PRECURSOR_FORMULAS)
    f.write(f"# {TARGET_FORMULA} Thermodynamic Reaction Network (v2)\n\n")
    f.write(f"**Method**: McDermott et al. Nat. Commun. 2021 graph-based approach\n")
    f.write(f"**Target**: {TARGET_FORMULA} ({target_id})\n")
    f.write(f"**Precursors**: {pfx}\n")
    f.write(f"**T_synthesis**: {T_SYNTHESIS} K\n")
    f.write(f"**Cost function**: C = ln(1 + (273/{T_SYNTHESIS}) * exp(Δg))\n\n")
    if EXPLORATION_MODE:
        f.write("## Candidate Reactions\n\n")
    else:
        f.write("## KSP Shortest Paths\n\n")
    if EXPLORATION_MODE:
        f.write("| 序号 | 反应方程式 | 成本 |\n")
        f.write("| :---: | :--- | :---: |\n")
        for i, rxn in enumerate(sorted_candidates[:20]):
            f.write(f"| {i+1} | {rxn} | {softplus_cost(rxn, T=T_SYNTHESIS):.4f} |\n")
    else:
        f.write("| 路径 | 总成本 | 步骤 | 反应方程式 | ΔG (eV/atom) | 步骤成本 |\n")
        f.write("| :---: | :---: | :---: | :--- | :---: | :---: |\n")
        for rank, (cost, path) in enumerate(ksp_paths[:5]):
            rxns_p, costs_p, dgs_p = extract_pathway(G, path)
            for j, rxn in enumerate(rxns_p):
                f.write(f"| {rank+1} | {cost:.3f} | {j+1} | {rxn} | {dgs_p[j]:+.3f} | {costs_p[j]:.3f} |\n")
    f.write("\n")
    if not EXPLORATION_MODE:
        f.write("## Decomposition Pathways\n\n")
        f.write("| 路径 | 总成本 | 步骤 | 反应方程式 | ΔG (eV/atom) | 步骤成本 |\n")
        f.write("| :---: | :---: | :---: | :--- | :---: | :---: |\n")
        for rank, (cost, path) in enumerate(ksp_dec[:3]):
            rxns_p, costs_p, dgs_p = extract_pathway(G_dec, path)
            for j, rxn in enumerate(rxns_p):
                f.write(f"| {rank+1} | {cost:.3f} | {j+1} | {rxn} | {dgs_p[j]:+.3f} | {costs_p[j]:.3f} |\n")
    f.write("\n")
    # --- Balanced Pathways (Linear Combination, 与论文 Tables 同口径) ---
    if "balanced_paths" in globals() and balanced_paths:
        f.write("## Balanced Pathways (Linear Combination)\n\n")
        f.write(f"Total balanced pathways: {len(balanced_paths)}\n\n")
        for pi, bp in enumerate(balanced_paths[:5]):
            f.write(f"### Path {pi+1} (weighted average cost = {bp.average_cost:.4f})\n\n")
            f.write("| 步骤 | 反应方程式 | ΔG (eV/atom) | 成本 | 倍数 |\n")
            f.write("| :---: | :--- | :---: | :---: | :---: |\n")
            for j, rxn in enumerate(bp.reactions):
                f.write(f"| {j+1} | {rxn} | {rxn.energy_per_atom:+.3f} | {bp.costs[j]:.3f} | {bp.coefficients[j]:.3f} |\n")
            f.write("\n")

print("\nkinetics_network_v2_report.md saved")
